In [1]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.model_selection import  train_test_split
import torch
from pathlib import Path


In [ ]:
#Mount Google Drive to access our dataset stored there instead of uploading files repeatedly
drive.mount('/content/drive')

In [ ]:
#Define the base directory where our Alzheimer's disease neuroimaging dataset is stored
BASE_DIR = '/content/drive/MyDrive/Final_Dataset1'

#Define folder structure based on our dataset organization:
# - Modality types: MRI, PET, co-registered (coreg)
# - Patient classes: Alzheimer's Disease (ad) vs Control Normal (cn)
# - Data augmentation status: original vs augmented

folders = [
    'original_mri_ad', 'original_mri_cn',
    'augmented_mri_ad', 'augmented_mri_cn',
    'original_pet_ad', 'original_pet_cn',
    'augmented_pet_ad', 'augmented_pet_cn',
    'coreg_ad_augmented', 'coreg_ad_original',
    'coreg_cn_augmented', 'coreg_cn_original'
]


In [ ]:
#Load neuroimaging data from NIfTI format which is the standard for brain imaging
def load_nifti_file(file_path):
    try:
        img = nib.load(file_path)
        #Convert to numpy array for analysis since we need numerical access to voxel values
        data = img.get_fdata()
        #Some scans might have time as 4th dimension; we only need spatial data for this analysis
        if len(data.shape) == 4:
            data = data[:, :, :, 0]
        return data, img.header
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

In [ ]:
#Build a catalogue of dataset entries to streamline loading, validation, and analysis without repeatedly scanning directories.
def datasetCatalogue(base_dir, folders):
    catalogue = []

    for folder in folders:
        folder_path = os.path.join(base_dir, folder)
        if not os.path.exists(folder_path):
        #Skip folders that don't exist to allow flexible folder management and avoid runtime errors.
            continue

        is_augmented = 'augmented' in folder #Identify augmented data early to enable separate evaluation if needed.
        is_coregistered = 'coreg' in folder  #Identify co-registered scans because they have different preprocessing assumptions.

        if is_coregistered:
            modality = 'coreg' #Treat co-registered data as a distinct modality for better downstream handling.
            label = 1 if 'ad' in folder else 0 #Infer labels from folder naming convention to avoid manual annotation.
        else:
            modality = 'mri' if 'mri' in folder else 'pet' #Default to standard modalities unless otherwise indicated.
            label = 1 if 'ad' in folder else 0


        nifti_files = [f for f in os.listdir(folder_path) if f.endswith('.nii.gz')]

        for nifti_file in nifti_files:
            file_path = os.path.join(folder_path, nifti_file)

            try:
                img = nib.load(file_path)
                shape = img.shape
                spacing = img.header.get_zooms()
            except Exception as e:
              #Continue processing other files even if some files are corrupt or unreadable, ensuring maximum dataset is used
                print(f"Error looking at {file_path}: {e}")
                continue

            #Consolidate relevant metadata to allow flexible querying, stratification, and data quality analysis later.
            catalogue.append({'file_path': file_path, 'filename': nifti_file, 'folder': folder, 'modality': modality, 'label': label, 'is_augmented': is_augmented, 'is_coregistered': is_coregistered, 'shape': shape,'spacing': spacing})

    df = pd.DataFrame(catalogue)
    return df

catalogue_df = datasetCatalogue(BASE_DIR, folders)

#Provide quick dataset health checks to detect class imbalance, missing modalities, or preprocessing issues early.
print(f"Total files: {len(catalogue_df)}")
print(f"MRI files: {len(catalogue_df[catalogue_df['modality'] == 'mri'])}")
print(f"PET files: {len(catalogue_df[catalogue_df['modality'] == 'pet'])}")
print(f"Co-registered files: {len(catalogue_df[catalogue_df['modality'] == 'coreg'])}")
print(f"AD samples: {len(catalogue_df[catalogue_df['label'] == 1])}")
print(f"CN samples: {len(catalogue_df[catalogue_df['label'] == 0])}")
print(f"Original samples: {len(catalogue_df[catalogue_df['is_augmented'] == False])}")
print(f"Augmented samples: {len(catalogue_df[catalogue_df['is_augmented'] == True])}")


print("\nShape distribution:")
print(catalogue_df.groupby(['modality', 'shape']).size())

In [ ]:
#Split the dataset into training, validation, and test sets while preserving label balance and ensuring reproducibility across splits.
def datasetSplitting(catalogue_df, test_size=0.2, val_size=0.15, random_state=42):

    #Use empty DataFrames to allow collection of splits across modalities
    train_df = pd.DataFrame()
    val_df = pd.DataFrame()
    test_df = pd.DataFrame()

    #Process each modality separately to avoid introducing modality bias during train/val/test splitting.
    for modality in ['mri', 'pet', 'coreg']:

        modality_df = catalogue_df[catalogue_df['modality'] == modality]  #Isolate modality-specific samples to treat structural and functional data appropriately.
        original_df = modality_df[~modality_df['is_augmented']]            #Only split original (non-augmented) data to maintain fair and unbiased validation and test sets.
        augmented_df = modality_df[modality_df['is_augmented']]            #Augmented data is reserved for training to enrich learning without contaminating evaluation.

        #Initial split to separate out a fixed percentage of the data for unbiased testing.
        train_val_df, test_mod_df = train_test_split(original_df, test_size=test_size, random_state=random_state, stratify=original_df['label'])  #Maintain the class ratio across splits to prevent skewed evaluation.

        #Adjust validation size relative to the remaining train_val pool after test samples have been removed.
        adjusted_val_size = val_size / (1 - test_size)

        #Secondary split to carve out a validation set from the remaining data, again enforcing class balance.
        train_mod_df, val_mod_df = train_test_split(train_val_df, test_size=adjusted_val_size, random_state=random_state, stratify=train_val_df['label'])

        #Augmented samples are intentionally injected into the training set only to improve model generalisation without biasing validation or test results.
        train_mod_df = pd.concat([train_mod_df, augmented_df])

        #Merge splits from each modality to construct the complete training, validation, and test datasets.
        train_df = pd.concat([train_df, train_mod_df])
        val_df = pd.concat([val_df, val_mod_df])
        test_df = pd.concat([test_df, test_mod_df])

    return train_df, val_df, test_df

train_df, val_df, test_df = datasetSplitting(catalogue_df)

#Provide quick visibility into label balance to detect possible stratification or split errors early.
print(f"Training set: {len(train_df)} files ({len(train_df[train_df['label'] == 1])} AD, {len(train_df[train_df['label'] == 0])} CN)")
print(f"Validation set: {len(val_df)} files ({len(val_df[val_df['label'] == 1])} AD, {len(val_df[val_df['label'] == 0])} CN)")
print(f"Test set: {len(test_df)} files ({len(test_df[test_df['label'] == 1])} AD, {len(test_df[test_df['label'] == 0])} CN)")

#Check the distribution of modalities per split to verify no modality has been inadvertently excluded or overrepresented.
print("\nModality distribution in training set:")
print(train_df.groupby(['modality', 'label']).size())
print("\nModality distribution in validation set:")
print(val_df.groupby(['modality', 'label']).size())
print("\nModality distribution in test set:")
print(test_df.groupby(['modality', 'label']).size())
